# Rimless wheel — from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/hybrid_boxmap/blob/main/notebooks/rimless_wheel_from_scratch.ipynb)

The system is given by

$$\ddot\theta=\sin\theta,$$

with guard and reset

$$G=\{(\alpha+\gamma,\dot\theta): \dot\theta\geq 0\},\qquad r(\theta,\dot\theta)=(2\gamma-\theta,\cos(2\alpha)\,\dot\theta),$$

where $\gamma=0.2$ and $2\alpha=0.8$.

Run all cells to construct the grid, sample the relation, compute the Morse graph and Conley indices, and save new records and figures. No stored relation or index is loaded. [Stored figures](rimless_wheel.ipynb).


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or newer is required; select a current Colab runtime.")

if "google.colab" in sys.modules:
    repo = Path("/content/hybrid_boxmap")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bernardorivas/hybrid_boxmap.git", str(repo)], check=True)
    # Installation also runs when the checkout already exists.
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "-e", str(repo),
        "cmgdb==1.3.3+fork.6", "--find-links",
        "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.6",
    ], check=True)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "code"]
    repo = next((p for p in candidates if (p / "demo/run_paper_examples.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Open this notebook from the hybrid_boxmap checkout.")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if shutil.which("dot") is None:
    raise RuntimeError("Install Graphviz (the dot executable) before plotting Morse graphs.")
import CMGDB
from hybrid_dynamics.src.carrier_kernel import cmgdb_provenance
if not hasattr(CMGDB, "ComputeCarrierChainMap"):
    raise RuntimeError("Install CMGDB 1.3.3+fork.6 as described in the README.")
print("Repository:", repo)
print("CMGDB:", cmgdb_provenance("native"))


## Phase portrait

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hybrid_dynamics.examples.paper_examples import paper_problem

problem = paper_problem("rimless-wheel")

fig, ax = plt.subplots(figsize=(5, 4))
for state in [(0.0, 0.05), (0.0, 0.2), (0.0, 0.9), (-0.2, 0.25), (0.3, -0.3), (0.3, -0.15), (-0.2, 1.0)]:
    trajectory = problem.system.simulate(np.array(state, float), (0.0, 10.0), max_jumps=30, max_step=0.01)
    color = None
    for segment in trajectory.segments:
        line, = ax.plot(*np.asarray(segment.state_values).T, lw=0.8, color=color)
        color = line.get_color()
ax.plot([0.6, 0.6], [0, 1], color="k", lw=2.5)  # guard
ax.set(xlim=(-0.2, 0.62), ylim=(-0.5, 1.0), xlabel="$\\theta$", ylabel="$\\dot\\theta$")
plt.show()

## Parameters

$\tau=0.5$; window $R=[-0.2,0.6]\times[-0.5,1]$; corners with padding and gap refinement depth 14; index map `auto`.

At level $n$ and offset $K$, the base grid has $2^{n+K}$ cells per axis and the phase grid has $2^{n+2}$ cells. The paper computation below uses $n=7$ and $K=0$: $128^2$ base cells and 512 phase cells. Changing $K$ preserves the phase grid; changing $n$ changes both.


In [ ]:
example = 'rimless-wheel'
tau, level, level_offset = 0.5, 7, 0
RUN_FINE = False  # True recomputes the 2048² comparison too (about 12 GB RAM).


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import Image, display

output_dir = Path("output") / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
output_dir.mkdir(parents=True, exist_ok=False)
print("Output:", output_dir.resolve())

import json
import time
import psutil

# These caps also work in a two-CPU Colab runtime.
def worker_counts(base_cells):
    available_gib = psutil.virtual_memory().available / 2**30
    cpus = os.cpu_count() or 1
    if base_cells >= 2048 and available_gib < 12:
        raise RuntimeError(
            f"Only {available_gib:.1f} GiB available. The 2048² computation needs about 12 GB; "
            "use a high-RAM Colab runtime and rerun from the setup cell."
        )
    workers = max(1, min(cpus, 8, int(available_gib // 2)))
    grid_copy_gib = max(0.25, (base_cells / 1024)**2)
    memory_cap = max(1, int(max(0, available_gib - 4) / grid_copy_gib))
    index_cap = 1 if base_cells >= 2048 else 2 if base_cells >= 1024 else 4
    return workers, max(1, min(workers, index_cap, memory_cap))

def recompute(example, tau, level, level_offset):
    base_cells, phase_cells = 2**(level + level_offset), 2**(level + 2)
    workers, index_workers = worker_counts(base_cells)
    run_dir = output_dir / (example + "-base" + str(base_cells) + "-" + uuid4().hex[:8])
    run_dir.mkdir(exist_ok=False)
    command = [
        sys.executable, "demo/run_paper_examples.py", example,
        "--tau", f"{example}={tau}", "--level", f"{example}={level}",
        "--level-offset", f"{example}={level_offset}",
        "--eval-mode", "corners", "--gap-refinement-depth", "14",
        "--workers", str(workers), "--index-workers", str(index_workers),
        "--index-map", "auto", "--conley-backend", "native",
        "--output-dir", str(run_dir),
    ]
    print(f"Computing {example}: base {base_cells}², phase {phase_cells}, "
          f"workers {workers}, index workers {index_workers}", flush=True)
    log = run_dir / "computation.log"
    env = os.environ.copy()
    env.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
    started = time.perf_counter()
    try:
        with log.open("w") as stream:
            subprocess.run(command, cwd=repo, env=env, stdout=stream, stderr=subprocess.STDOUT, check=True)
    except subprocess.CalledProcessError:
        print(log.read_text()[-12000:])
        print("Full log:", log.resolve())
        raise
    wall_seconds = time.perf_counter() - started
    records = list(run_dir.glob("paper-*.json"))
    if len(records) != 1:
        raise RuntimeError(f"Expected one new computation record in {run_dir}, found {len(records)}.")
    print("Log:", log.resolve())
    return records[0], wall_seconds


## Compute

Each run gets a new directory under `output/`. The log and JSON record retain the command, parameters, timing, and any blocked index or `auto` fallback.

In [ ]:
record, wall_seconds = recompute(example, tau, level, level_offset)


In [ ]:
import json
from hybrid_dynamics.examples.paper_figures import conley_index_dimensions

def report_record(record, wall_seconds=None):
    summary = json.loads(Path(record).read_text())
    variant = summary["figure_variants"]["nontrivial"]
    print(f"Base: {summary['base_cells_per_axis']}²; phase cells: {summary['phase_cells']}; "
          f"Morse sets: {len(summary['morse_graph']['nodes'])}")
    print("Restricted order:", variant["order_shown"])
    if wall_seconds is not None:
        print(f"Wall time: {wall_seconds:.2f} s")
    print("Node | index (degrees 0, 1, 2) | construction")
    for entry in summary["conley"]:
        if entry["computed"]:
            dimensions = conley_index_dimensions(entry["shift_class"])
            label = dimensions[:3] if dimensions is not None else entry["shift_class"]
        else:
            label = "BLOCKED: " + entry.get("blocker", "see record")
        print(entry["morse_node"], "|", label, "|", entry.get("label_source", ""))
        if entry.get("exit_components_blocker"):
            print("  auto fallback:", entry["exit_components_blocker"])
    print("Record:", Path(record).resolve())
    return summary

summary = report_record(record, wall_seconds)


## Figures

Labels give index dimensions in degrees 0, 1, 2. These are indices of the computed finite relation. All figures below use the new record.

In [ ]:
from demo.paper_comparison import render_wheel, render_wheel_comparison

figures = render_wheel(record, record.parent / "panels", titles=False)
display(Image(figures["comparison_files"][1], width=1100))
display(Image(str(record.with_name(record.stem + "-graph.png")), width=520))


## Optional fine comparison

Set `RUN_FINE = True` above to recompute $2048^2$ base cells with the same 512 phase cells. This requires about 12 GB of memory. It reads no stored fine record.

In [ ]:
if RUN_FINE:
    fine_record, fine_wall_seconds = recompute(example, tau, level=7, level_offset=4)
    fine_summary = report_record(fine_record, fine_wall_seconds)
    comparison = render_wheel_comparison(record, fine_record, output_dir / "comparison", titles=False)
    display(Image(comparison["comparison_files"][1], width=1100))
    display(Image(comparison["full_graph_comparison_files"][1], width=900))
else:
    print("Fine recomputation skipped; set RUN_FINE = True to compute both resolutions.")


In [ ]:
print("All new outputs:")
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(path.resolve())
